# 02 — Baseline RAG

**VerifiedRAG — Multi-Agent RAG with Claim-Level Fact Verification**

This notebook implements the standard RAG baseline that later verified/agentic systems will be compared against.

```text
Question → Query Embedding → FAISS → Top-k Evidence → Prompt → Local LLM → Answer + Citations
```

The corpus and FAISS index are loaded from `01_data_preparation.ipynb`, keeping the retrieval conditions fixed for fair experiments.

## 1. Install dependencies

Designed for Google Colab. A GPU is strongly recommended for local LLM generation.

In [1]:
!pip -q install transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.6 MB/s eta 0:00:00


In [2]:
# 1. Clone your repository to get the project files
!git clone https://github.com/faizzanasghar/Verified-RAG.git /content/Verified-RAG

# 2. Run the data preparation notebook in this runtime to build the data and FAISS index
%run /content/Verified-RAG/notebooks/01_data_preparation.ipynb

Cloning into '/content/Verified-RAG'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 26 (delta 3), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 897.76 KiB | 5.95 MiB/s, done.
Resolving deltas: 100% (3/3), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 54.8 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Project root: /content/Verified-RAG
Raw PDFs: /content/Verified-RAG/data/raw_pdfs
Processed data: /content/Verified-RAG/data/processed
Vector store: /content/Verified-RAG/data/vector_store


,paper_id,title,path,status,size_bytes
0,attention_2017,Attention Is All You Need,/content/Verified-RAG/data/raw_pdfs/attention_...,downloaded,2215244
1,resnet_2015,Deep Residual Learning for Image Recognition,/content/Verified-RAG/data/raw_pdfs/resnet_201...,downloaded,819383
2,bert_2018,BERT: Pre-training of Deep Bidirectional Trans...,/content/Verified-RAG/data/raw_pdfs/bert_2018.pdf,downloaded,775166
3,adam_2014,Adam: A Method for Stochastic Optimization,/content/Verified-RAG/data/raw_pdfs/adam_2014.pdf,downloaded,584641
4,gpt2_2019,Language Models are Unsupervised Multitask Lea...,/content/Verified-RAG/data/raw_pdfs/gpt2_2019.pdf,downloaded,582775


adam_2014.pdf                                 570.9 KB
attention_2017.pdf                            2163.3 KB
bert_2018.pdf                                 757.0 KB
gpt2_2019.pdf                                 569.1 KB
resnet_2015.pdf                               800.2 KB


Extracting PDFs:   0%|          | 0/5 [00:00<?, ?it/s]

Extracted pages: 82


...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda
Embedding dimension: 384


/tmp/ipykernel_4077/335280999.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding matrix shape: (516, 384)
FAISS index type: IndexFlatIP
Vectors in index: 516
Vector dimension: 384
Saved:
/content/Verified-RAG/data/vector_store/faiss.index
/content/Verified-RAG/data/vector_store/chunks_metadata.json
/content/Verified-RAG/data/vector_store/index_metadata.json
Rank: 1
Score: 0.6450
Paper: Deep Residual Learning for Image Recognition
Page: 1
Chunk ID: resnet_2015_p001_c000
Deep Residual Learning for Image Recognition Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun Microsoft Research {kahe, v-xiangz, v-shren, jiansun}@microsoft.com Abstract Deeper neural networks are more difﬁcult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to op

## 2. Imports and project paths

In [3]:
from pathlib import Path
import json
import re
import time
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

PROJECT_ROOT = Path("/content/Verified-RAG")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)


Project: /content/Verified-RAG


## 3. Check the Colab runtime

In [4]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("WARNING: No GPU detected. LLM inference may be very slow.")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


## 4. Load the corpus and FAISS index

In [5]:
required = [
    PROCESSED_DIR / "chunks.jsonl",
    PROCESSED_DIR / "papers.json",
    PROCESSED_DIR / "corpus_metadata.json",
    VECTOR_DIR / "faiss.index",
    VECTOR_DIR / "chunks_metadata.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run 01_data_preparation.ipynb first. Missing:\n" + "\n".join(missing))

with (PROCESSED_DIR / "corpus_metadata.json").open(encoding="utf-8") as f:
    corpus_metadata = json.load(f)
with (PROCESSED_DIR / "papers.json").open(encoding="utf-8") as f:
    papers = json.load(f)
with (VECTOR_DIR / "chunks_metadata.json").open(encoding="utf-8") as f:
    chunks_metadata = json.load(f)

faiss_index = faiss.read_index(str(VECTOR_DIR / "faiss.index"))

print("Papers:", len(papers))
print("Chunks:", len(chunks_metadata))
print("FAISS vectors:", faiss_index.ntotal)
print(json.dumps(corpus_metadata, indent=2))


Papers: 5
Chunks: 516
FAISS vectors: 516
{
  "num_papers": 5,
  "num_pages": 82,
  "num_chunks": 516,
  "chunk_size": 1000,
  "chunk_overlap": 150,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
}


## 5. Load the same embedding model used during data preparation

In [6]:
EMBEDDING_MODEL_NAME = corpus_metadata.get(
    "embedding_model", "sentence-transformers/all-MiniLM-L6-v2"
)
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=embedding_device)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Device:", embedding_device)
print("Dimension:", embedding_model.get_sentence_embedding_dimension())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda
Dimension: 384


/tmp/ipykernel_4077/4284008764.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimension:", embedding_model.get_sentence_embedding_dimension())


## 6. Retriever

In [7]:
DEFAULT_TOP_K = 5

def retrieve(query: str, top_k: int = DEFAULT_TOP_K, min_score=None):
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        score = float(score)
        if min_score is not None and score < min_score:
            continue
        item = chunks_metadata[int(idx)].copy()
        item["score"] = score
        results.append(item)
    return results

def display_retrieval_results(results, max_chars=1000):
    for rank, item in enumerate(results, 1):
        print("=" * 100)
        print(f"Rank: {rank}")
        print(f"Score: {item['score']:.4f}")
        print(f"Paper: {item['title']}")
        print(f"Page: {item['page']}")
        print(f"Chunk: {item['chunk_id']}")
        print("-" * 100)
        print(item["text"][:max_chars])
        print()

test_results = retrieve(
    "What problem does residual learning solve in very deep neural networks?", 5
)
display_retrieval_results(test_results)


Rank: 1
Score: 0.6674
Paper: Deep Residual Learning for Image Recognition
Page: 1
Chunk: resnet_2015_p001_c000
----------------------------------------------------------------------------------------------------
Deep Residual Learning for Image Recognition Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun Microsoft Research {kahe, v-xiangz, v-shren, jiansun}@microsoft.com Abstract Deeper neural networks are more difﬁcult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers—8× deeper than VGG nets [41] but still havi

## 7. Load the local open-source LLM

Default model: `Qwen/Qwen2.5-3B-Instruct`. A 4-bit load is used on CUDA to reduce VRAM usage.

In [8]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if torch.cuda.is_available():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
else:
    print("CPU fallback selected; inference can be slow.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

model.eval()
print("Loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


## 8. Build the RAG prompt

In [9]:
SYSTEM_PROMPT = """You are a technical question-answering assistant.

Answer the user's question using ONLY the supplied source context.

Rules:
1. Do not invent facts that are not supported by the context.
2. If the context is insufficient, say so.
3. Every substantive factual claim should include a source citation.
4. Citations must use exactly: [SOURCE: chunk_id]
5. Do not create or modify source IDs.
6. Be concise and technically precise.
"""

def build_context(results):
    blocks = []
    for item in results:
        blocks.append(
            f"[SOURCE: {item['chunk_id']}]\n"
            f"Paper: {item['title']}\n"
            f"Page: {item['page']}\n"
            f"Text:\n{item['text']}"
        )
    return "\n\n".join(blocks)

def build_rag_prompt(question, results):
    return f"""Answer the following question using only the source context.

QUESTION:
{question}

SOURCE CONTEXT:
{build_context(results)}

Remember:
- Every substantive factual claim must have a [SOURCE: chunk_id] citation.
- If the context is insufficient, explicitly state that.
- Do not use information that is not supported by the supplied context.
"""


## 9. LLM generation

In [10]:
def generate_answer(question, retrieved_results, max_new_tokens=350):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_rag_prompt(question, retrieved_results)},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text, return_tensors="pt", truncation=True, max_length=8192
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    start = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start

    input_len = inputs["input_ids"].shape[1]
    generated = outputs[0][input_len:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "answer": answer,
        "generation_time_seconds": elapsed,
        "prompt_tokens": int(input_len),
        "generated_tokens": int(len(generated)),
    }


## 10. Complete baseline RAG pipeline

In [11]:
def baseline_rag(question, top_k=5, min_score=None, max_new_tokens=350):
    retrieval_start = time.perf_counter()
    retrieved = retrieve(question, top_k=top_k, min_score=min_score)
    retrieval_time = time.perf_counter() - retrieval_start

    if not retrieved:
        return {
            "question": question,
            "answer": "I could not retrieve sufficiently relevant evidence.",
            "sources": [],
            "retrieved_chunks": [],
            "retrieval_time_seconds": retrieval_time,
            "generation_time_seconds": 0.0,
            "prompt_tokens": 0,
            "generated_tokens": 0,
        }

    generation = generate_answer(question, retrieved, max_new_tokens)

    return {
        "question": question,
        "answer": generation["answer"],
        "sources": [
            {
                "chunk_id": r["chunk_id"],
                "paper_id": r["paper_id"],
                "title": r["title"],
                "page": r["page"],
                "score": r["score"],
            }
            for r in retrieved
        ],
        "retrieved_chunks": retrieved,
        "retrieval_time_seconds": retrieval_time,
        "generation_time_seconds": generation["generation_time_seconds"],
        "prompt_tokens": generation["prompt_tokens"],
        "generated_tokens": generation["generated_tokens"],
    }


## 11. Run one baseline RAG query

In [12]:
question = (
    "What is the main idea behind residual learning, "
    "and why does it help train very deep neural networks?"
)
result = baseline_rag(question, top_k=5, max_new_tokens=300)

print("QUESTION\n", result["question"])
print("\nANSWER\n", result["answer"])
print("\nSOURCES")
for source in result["sources"]:
    print(
        f"- {source['chunk_id']} | {source['title']} | "
        f"page {source['page']} | score={source['score']:.4f}"
    )
print("\nTIMING")
print("Retrieval:", round(result["retrieval_time_seconds"], 3), "s")
print("Generation:", round(result["generation_time_seconds"], 3), "s")


QUESTION
 What is the main idea behind residual learning, and why does it help train very deep neural networks?

ANSWER
 The main idea behind residual learning is to explicitly learn residual functions instead of unreferenced functions. This approach helps train very deep neural networks by making the optimization process easier. By casting the original function as \( F(x) + x \), the residual learning framework allows the network to focus on fitting the residual mapping rather than the original mapping directly. This makes the training process smoother and enables deeper networks to achieve better performance [SOURCE: resnet_2015_p001_c000].

Residual learning helps train very deep neural networks because it simplifies the optimization process. As networks become deeper, residual learning allows them to gain accuracy from increased depth, which is not possible with traditional methods [SOURCE: resnet_2015_p002_c002]. The framework shows that residual networks can be 8 times deeper tha

## 12. Citation diagnostics

In [13]:
SOURCE_PATTERN = re.compile(r"\[SOURCE:\s*([^\]]+)\]", flags=re.IGNORECASE)

def extract_citations(answer):
    return [x.strip() for x in SOURCE_PATTERN.findall(answer)]

def citation_diagnostics(result):
    cited = extract_citations(result["answer"])
    retrieved_ids = {x["chunk_id"] for x in result["sources"]}
    return {
        "citations": cited,
        "valid_citations": [x for x in cited if x in retrieved_ids],
        "invalid_citations": [x for x in cited if x not in retrieved_ids],
    }

print(json.dumps(citation_diagnostics(result), indent=2))


{
  "citations": [
    "resnet_2015_p001_c000",
    "resnet_2015_p002_c002",
    "resnet_2015_p001_c000"
  ],
  "valid_citations": [
    "resnet_2015_p001_c000",
    "resnet_2015_p002_c002",
    "resnet_2015_p001_c000"
  ],
  "invalid_citations": []
}


## 13. Inspect the exact evidence available to the generator

In [14]:
for rank, item in enumerate(result["retrieved_chunks"], 1):
    print("=" * 100)
    print(f"SOURCE #{rank}: {item['chunk_id']}")
    print(f"Paper: {item['title']} | Page: {item['page']} | Score: {item['score']:.4f}")
    print("-" * 100)
    print(item["text"])
    print()


SOURCE #1: resnet_2015_p001_c000
Paper: Deep Residual Learning for Image Recognition | Page: 1 | Score: 0.5869
----------------------------------------------------------------------------------------------------
Deep Residual Learning for Image Recognition Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun Microsoft Research {kahe, v-xiangz, v-shren, jiansun}@microsoft.com Abstract Deeper neural networks are more difﬁcult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unreferenced functions. We provide comprehensive empirical evidence showing that these residual networks are easier to optimize, and can gain accuracy from considerably increased depth. On the ImageNet dataset we evaluate residual nets with a depth of up to 152 layers—8× deeper than VGG nets [41] but still havi

## 14. Small baseline benchmark

This is a pipeline sanity-check benchmark, not the final research evaluation.

In [15]:
BASELINE_QUESTIONS = [
    {
        "id": "q01",
        "question": "What problem does residual learning address in very deep neural networks?",
        "expected_papers": ["resnet_2015"],
    },
    {
        "id": "q02",
        "question": "What is the main architectural idea of the Transformer?",
        "expected_papers": ["attention_2017"],
    },
    {
        "id": "q03",
        "question": "Why does the Transformer not require recurrence to model relationships between tokens?",
        "expected_papers": ["attention_2017"],
    },
    {
        "id": "q04",
        "question": "What is the purpose of pre-training in BERT?",
        "expected_papers": ["bert_2018"],
    },
    {
        "id": "q05",
        "question": "What are the two moment estimates used by Adam?",
        "expected_papers": ["adam_2014"],
    },
    {
        "id": "q06",
        "question": "How does Adam adapt the learning rate for individual parameters?",
        "expected_papers": ["adam_2014"],
    },
]
display(pd.DataFrame(BASELINE_QUESTIONS))


,id,question,expected_papers
0,q01,What problem does residual learning address in...,[resnet_2015]
1,q02,What is the main architectural idea of the Tra...,[attention_2017]
2,q03,Why does the Transformer not require recurrenc...,[attention_2017]
3,q04,What is the purpose of pre-training in BERT?,[bert_2018]
4,q05,What are the two moment estimates used by Adam?,[adam_2014]
5,q06,How does Adam adapt the learning rate for indi...,[adam_2014]


## 15. Run the benchmark

In [16]:
baseline_outputs = []

for item in tqdm(BASELINE_QUESTIONS, desc="Running baseline RAG"):
    output = baseline_rag(item["question"], top_k=5, max_new_tokens=300)
    diagnostics = citation_diagnostics(output)

    baseline_outputs.append({
        "question_id": item["id"],
        "question": item["question"],
        "expected_papers": item["expected_papers"],
        "answer": output["answer"],
        "sources": output["sources"],
        "retrieved_chunks": output["retrieved_chunks"],
        "retrieval_time_seconds": output["retrieval_time_seconds"],
        "generation_time_seconds": output["generation_time_seconds"],
        "prompt_tokens": output["prompt_tokens"],
        "generated_tokens": output["generated_tokens"],
        "citations": diagnostics["citations"],
        "valid_citations": diagnostics["valid_citations"],
        "invalid_citations": diagnostics["invalid_citations"],
    })

print("Completed:", len(baseline_outputs))


Running baseline RAG:   0%|          | 0/6 [00:00<?, ?it/s]

Completed: 6


## 16. Review generated answers

In [17]:
for record in baseline_outputs:
    print("=" * 110)
    print(f"{record['question_id']}: {record['question']}")
    print("-" * 110)
    print(record["answer"])
    print("\nCitations:", record["citations"])
    print("Invalid citations:", record["invalid_citations"])
    print()


q01: What problem does residual learning address in very deep neural networks?
--------------------------------------------------------------------------------------------------------------
Residual learning addresses the problem of vanishing/exploding gradients in very deep neural networks, which hinders their convergence. As networks become deeper, residual learning helps prevent degradation in performance by allowing the network to learn residual functions that help maintain accuracy even as depth increases. This approach mitigates the issue of overfitting and allows for the training of very deep networks that achieve better performance than shallower networks. [SOURCE: resnet_2015_p001_c003]

Citations: ['resnet_2015_p001_c003']
Invalid citations: []

q02: What is the main architectural idea of the Transformer?
--------------------------------------------------------------------------------------------------------------
The main architectural idea of the Transformer is the use of s

## 17. Basic baseline diagnostics

In [18]:
rows = []
for record in baseline_outputs:
    rows.append({
        "question_id": record["question_id"],
        "num_sources": len(record["sources"]),
        "num_citations": len(record["citations"]),
        "num_valid_citations": len(record["valid_citations"]),
        "num_invalid_citations": len(record["invalid_citations"]),
        "has_citation": bool(record["citations"]),
        "has_invalid_citation": bool(record["invalid_citations"]),
        "retrieval_time_seconds": record["retrieval_time_seconds"],
        "generation_time_seconds": record["generation_time_seconds"],
        "prompt_tokens": record["prompt_tokens"],
        "generated_tokens": record["generated_tokens"],
    })

baseline_metrics_df = pd.DataFrame(rows)
display(baseline_metrics_df)

print("Citation presence:",
      round(baseline_metrics_df["has_citation"].mean() * 100, 2), "%")
print("Invalid citation rate:",
      round(baseline_metrics_df["has_invalid_citation"].mean() * 100, 2), "%")
print("Mean retrieval time:",
      round(baseline_metrics_df["retrieval_time_seconds"].mean(), 3), "s")
print("Mean generation time:",
      round(baseline_metrics_df["generation_time_seconds"].mean(), 3), "s")


,question_id,num_sources,num_citations,num_valid_citations,num_invalid_citations,has_citation,has_invalid_citation,retrieval_time_seconds,generation_time_seconds,prompt_tokens,generated_tokens
0,q01,5,1,1,0,True,False,0.017638,7.328718,1136,102
1,q02,5,1,1,0,True,False,0.010467,9.758369,1005,129
2,q03,5,4,4,0,True,False,0.014494,15.271464,1029,210
3,q04,5,4,4,0,True,False,0.010353,10.809471,1075,144
4,q05,5,2,2,0,True,False,0.010304,5.041092,1147,59
5,q06,5,1,1,0,True,False,0.012491,14.108763,1294,187


Citation presence: 100.0 %
Invalid citation rate: 0.0 %
Mean retrieval time: 0.013 s
Mean generation time: 10.386 s


## 18. Expected-paper retrieval hit rate

In [19]:
def retrieval_paper_hit(record):
    retrieved_papers = {s["paper_id"] for s in record["sources"]}
    return bool(retrieved_papers.intersection(record["expected_papers"]))

coverage = pd.DataFrame([
    {
        "question_id": r["question_id"],
        "expected_papers": r["expected_papers"],
        "retrieval_hit": retrieval_paper_hit(r)
    }
    for r in baseline_outputs
])

display(coverage)
print(
    "Expected-paper retrieval hit rate:",
    round(coverage["retrieval_hit"].mean() * 100, 2), "%"
)


,question_id,expected_papers,retrieval_hit
0,q01,[resnet_2015],True
1,q02,[attention_2017],True
2,q03,[attention_2017],True
3,q04,[bert_2018],True
4,q05,[adam_2014],True
5,q06,[adam_2014],True


Expected-paper retrieval hit rate: 100.0 %


## 19. Save the baseline experiment

In [20]:
experiment_name = "baseline_rag_v1"

metadata = {
    "experiment": experiment_name,
    "model": MODEL_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "top_k": 5,
    "max_new_tokens": 300,
    "sampling": False,
    "temperature": 0.0,
    "corpus_metadata": corpus_metadata,
    "num_questions": len(baseline_outputs),
    "runtime": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else "CPU"
    ),
}

payload = {"metadata": metadata, "results": baseline_outputs}

json_path = RESULTS_DIR / f"{experiment_name}.json"
csv_path = RESULTS_DIR / f"{experiment_name}_metrics.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

baseline_metrics_df.to_csv(csv_path, index=False)

print("Saved:", json_path)
print("Saved:", csv_path)


Saved: /content/Verified-RAG/results/baseline_rag_v1.json
Saved: /content/Verified-RAG/results/baseline_rag_v1_metrics.csv


# Baseline RAG complete

We now have:

```text
Question
   ↓
MiniLM embedding
   ↓
FAISS
   ↓
Top-k evidence
   ↓
Qwen2.5-3B-Instruct
   ↓
Cited answer
```

The baseline does **not** independently verify whether claims are supported. It does not perform claim decomposition, evidence checking, contradiction detection, or regeneration.

Those capabilities will be introduced in `03_multi_agent_rag.ipynb`.

The saved baseline results become the **control condition** for the research experiments.

In [ ]:
import json
from pathlib import Path
from google.colab import _message

# 1. Download current notebook JSON into the repo's notebooks/ folder
notebook_json = _message.blocking_request("get_ipynb")
target_nb_path = Path("/content/Verified-RAG/notebooks/02_baseline_rag.ipynb")

with open(target_nb_path, "w", encoding="utf-8") as f:
    json.dump(notebook_json["ipynb"], f, indent=2)

print(f"Saved notebook to {target_nb_path}")